# Mars BEACON — T4 static terrain preview
120 calibrated synthetic views of measured Gale orbital terrain. 96 train / 12 validation / 12 reserved test. This is orbital distillation, not rover reconstruction. Select a T4 runtime. Upload the supplied preview bundle.


In [ ]:
import sys, subprocess, os
os.environ['MAX_JOBS']='2'
os.environ['TORCH_CUDA_ARCH_LIST']='7.5'
subprocess.check_call([sys.executable,'-m','pip','install','-q','gsplat==1.5.3','ninja','plyfile','scikit-image'])
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0),torch.__version__,torch.version.cuda)


In [ ]:
from google.colab import files
files.upload() # Select Mars_BEACON_Terrain_Preview_Bundle.zip


In [ ]:
import pathlib,zipfile,hashlib
root=pathlib.Path('/content/mars_beacon_terrain')
bundle=pathlib.Path('/content/Mars_BEACON_Terrain_Preview_Bundle.zip')
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='b752a1d4149798bbb6066b29bb413e136199566b8f6e93f7846cd2a1f86537cf'
with zipfile.ZipFile(bundle) as z:
    assert z.testzip() is None
    z.extractall(root)


In [ ]:
cmd=[sys.executable,'-u',str(root/'scripts/train_gsplat.py'),'--data',str(root/'data/gale_preview'),'--out',str(root/'runs/preview'),'--steps','5000']
with subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
    for line in proc.stdout: print(line,end='',flush=True)
    assert proc.wait()==0, 'Training failed; inspect output above'


In [ ]:
import json
from IPython.display import display,Image
print((root/'runs/preview/validation_005000.json').read_text())
for p in sorted((root/'runs/preview').glob('val_005000_*.png')): display(Image(filename=str(p)))
subprocess.check_call([sys.executable,str(root/'scripts/geometry_report.py'),str(root/'runs/preview/point_cloud_005000.ply'),str(root/'data/gale_preview'),str(root/'runs/preview/geometry.json')])


## Gate before 30k, temporal training and browser delivery
Review held-out renders, geometry report and Gaussian PLY. Keep the final test split untouched during tuning. No automatic 30k run. Environmental prototypes are in the local physics folder; calibrated temperature, radiation, rover interaction and independent browser performance remain separate gates.


In [ ]:
import shutil
archive=shutil.make_archive('/content/Mars_BEACON_preview_results','zip',root/'runs/preview')
files.download(archive) # Save before runtime disconnects
